## 1. Setup and Imports

In [1]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().absolute().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Project imports
from config.config import Config
from src.data_loader import StockDataLoader
from src.data_preprocessor import StockDataPreprocessor
from src.model import LSTMStockModel

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
print("✓ Imports successful")

TensorFlow version: 2.19.0
GPU available: []
✓ Imports successful


## 2. Configuration

In [2]:
# Initialize configuration
config = Config()

# Enhanced settings
config.preprocessing.sequence_length = 30
config.model.lstm_units = [128, 64, 32]
config.model.dropout_rate = 0.3
config.model.learning_rate = 0.001
config.training.epochs = 100
config.training.batch_size = 32

print("Configuration loaded successfully")

Configuration loaded successfully


## 3. Load Raw Data (DON'T PROCESS YET!)

In [3]:
# Load data
data_path = project_root / config.data.raw_data_path
print(f"Loading data from: {data_path}")

data_loader = StockDataLoader(str(data_path))
df = data_loader.load_csv(date_column=config.data.date_column)

print(f"\n✓ Loaded {len(df):,} records")
print(f"✓ Date range: {df[config.data.date_column].min()} to {df[config.data.date_column].max()}")

# Create preprocessor
preprocessor = StockDataPreprocessor(
    feature_columns=config.data.feature_columns,
    target_column=config.data.target_column
)

# Only handle missing values (no feature engineering yet!)
df = preprocessor.handle_missing_values(df, method='ffill')

print(f"\n✓ Raw data shape: {df.shape}")
df.head()

INFO:src.data_loader:Loaded 245 records from c:\Users\USER\OneDrive\Documents\GitHub\Research-Project\backend\lstm_stock_prediction\data\processed\lstm_ready_data.csv


Loading data from: c:\Users\USER\OneDrive\Documents\GitHub\Research-Project\backend\lstm_stock_prediction\data\processed\lstm_ready_data.csv

✓ Loaded 245 records
✓ Date range: 2000-01-01 00:00:00 to 2020-06-01 00:00:00

✓ Raw data shape: (245, 6)


,Date,Open,High,Low,Close,Volume
0,2000-01-01,102.860335,43.043575,40.854469,102.860335,18412.0
1,2000-02-01,82.811518,40.917801,38.652880,82.827225,15820.0
2,2000-03-01,71.360406,34.579695,31.312690,71.375635,14061.0
3,2000-04-01,38.967033,33.376923,31.453846,39.368132,7165.0
4,2000-05-01,76.277778,36.832500,33.632778,76.477778,13766.0


## 4. 🚨 STEP 1: Split Data FIRST (Critical for Time Series!)

In [4]:
# Calculate split indices (time-based split)
total_samples = len(df)
train_idx = int(total_samples * 0.8)
val_idx = train_idx + int(total_samples * 0.1)

# Split the raw data by time
df_train_raw = df.iloc[:train_idx].copy()
df_val_raw = df.iloc[train_idx:val_idx].copy()
df_test_raw = df.iloc[val_idx:].copy()

print("=" * 70)
print("CRITICAL: DATA SPLIT BEFORE PROCESSING")
print("=" * 70)
print(f"\n📅 Training: {len(df_train_raw):,} samples ({len(df_train_raw)/total_samples*100:.1f}%)")
print(f"   Dates: {df_train_raw[config.data.date_column].min()} → {df_train_raw[config.data.date_column].max()}")

print(f"\n📅 Validation: {len(df_val_raw):,} samples ({len(df_val_raw)/total_samples*100:.1f}%)")
print(f"   Dates: {df_val_raw[config.data.date_column].min()} → {df_val_raw[config.data.date_column].max()}")

print(f"\n📅 Test: {len(df_test_raw):,} samples ({len(df_test_raw)/total_samples*100:.1f}%)")
print(f"   Dates: {df_test_raw[config.data.date_column].min()} → {df_test_raw[config.data.date_column].max()}")

print("\n✅ Data split by TIME - no overlap between sets!")

CRITICAL: DATA SPLIT BEFORE PROCESSING

📅 Training: 196 samples (80.0%)
   Dates: 2000-01-01 00:00:00 → 2016-04-01 00:00:00

📅 Validation: 24 samples (9.8%)
   Dates: 2016-05-01 00:00:00 → 2018-04-01 00:00:00

📅 Test: 25 samples (10.2%)
   Dates: 2018-05-01 00:00:00 → 2020-06-01 00:00:00

✅ Data split by TIME - no overlap between sets!


## 5. 🔧 STEP 2: Add Technical Indicators (with Rolling Window Buffer)

**Critical:** We need historical data to calculate indicators properly.  
We'll use a rolling window approach to avoid losing validation/test samples.

In [5]:
# Define lookback window for technical indicators (max window needed)
indicator_lookback = 60  # Need at least 60 days for 30-day MA + RSI calculations

print("=" * 70)
print("ADDING TECHNICAL INDICATORS WITH ROLLING WINDOW")
print("=" * 70)

# Store the boundary dates for proper slicing after indicators
val_start_date = df_val_raw[config.data.date_column].iloc[0]
test_start_date = df_test_raw[config.data.date_column].iloc[0]

# For training set: just add indicators directly
print(f"\n1. Processing TRAINING set...")
df_train = preprocessor.add_technical_indicators(df_train_raw)
print(f"   Training set after indicators: {df_train.shape}")

# For validation set: include buffer from training for proper calculation
print(f"\n2. Processing VALIDATION set with buffer...")
# Get last 'indicator_lookback' days from training as buffer
val_start_idx = max(0, train_idx - indicator_lookback)
df_val_with_buffer = df.iloc[val_start_idx:val_idx].copy()
print(f"   Buffer + Validation combined: {df_val_with_buffer.shape}")

# Add indicators to combined data
df_val_with_indicators = preprocessor.add_technical_indicators(df_val_with_buffer)
# Keep only rows with dates >= validation start date
df_val = df_val_with_indicators[
    df_val_with_indicators[config.data.date_column] >= val_start_date
].copy()
print(f"   Validation set after removing buffer: {df_val.shape}")

# For test set: include buffer from training+validation for proper calculation
print(f"\n3. Processing TEST set with buffer...")
test_start_idx = max(0, val_idx - indicator_lookback)
df_test_with_buffer = df.iloc[test_start_idx:].copy()
print(f"   Buffer + Test combined: {df_test_with_buffer.shape}")

# Add indicators to combined data
df_test_with_indicators = preprocessor.add_technical_indicators(df_test_with_buffer)
# Keep only rows with dates >= test start date
df_test = df_test_with_indicators[
    df_test_with_indicators[config.data.date_column] >= test_start_date
].copy()
print(f"   Test set after removing buffer: {df_test.shape}")

print(f"\n{'='*70}")
print("FINAL SHAPES AFTER INDICATORS:")
print(f"{'='*70}")
print(f"✅ Training set: {df_train.shape}")
print(f"✅ Validation set: {df_val.shape}")
print(f"✅ Test set: {df_test.shape}")
print(f"\n✅ Total features: {len(df_train.columns)}")

# Verify we have enough samples for sequence creation
print(f"\n🔍 Checking data sufficiency for sequence_length={config.preprocessing.sequence_length}:")
if len(df_train) > config.preprocessing.sequence_length:
    print(f"   ✅ Training: {len(df_train)} rows (sufficient)")
else:
    print(f"   ❌ Training: {len(df_train)} rows (INSUFFICIENT!)")
    
if len(df_val) > config.preprocessing.sequence_length:
    print(f"   ✅ Validation: {len(df_val)} rows (sufficient)")
else:
    print(f"   ⚠️ Validation: {len(df_val)} rows (may be insufficient)")
    
if len(df_test) > config.preprocessing.sequence_length:
    print(f"   ✅ Test: {len(df_test)} rows (sufficient)")
else:
    print(f"   ⚠️ Test: {len(df_test)} rows (may be insufficient)")

print(f"\n✅ All sets have proper technical indicators!")
print(f"✅ NO data leakage - only past data used for indicators!")

ADDING TECHNICAL INDICATORS WITH ROLLING WINDOW

1. Processing TRAINING set...


INFO:src.data_preprocessor:Added comprehensive technical indicators. New shape: (167, 41)
INFO:src.data_preprocessor:Total features: 41
INFO:src.data_preprocessor:Added comprehensive technical indicators. New shape: (55, 41)


   Training set after indicators: (167, 41)

2. Processing VALIDATION set with buffer...
   Buffer + Validation combined: (84, 6)


INFO:src.data_preprocessor:Total features: 41
INFO:src.data_preprocessor:Added comprehensive technical indicators. New shape: (56, 41)
INFO:src.data_preprocessor:Total features: 41


   Validation set after removing buffer: (24, 41)

3. Processing TEST set with buffer...
   Buffer + Test combined: (85, 6)
   Test set after removing buffer: (25, 41)

FINAL SHAPES AFTER INDICATORS:
✅ Training set: (167, 41)
✅ Validation set: (24, 41)
✅ Test set: (25, 41)

✅ Total features: 41

🔍 Checking data sufficiency for sequence_length=30:
   ✅ Training: 167 rows (sufficient)
   ⚠️ Validation: 24 rows (may be insufficient)
   ⚠️ Test: 25 rows (may be insufficient)

✅ All sets have proper technical indicators!
✅ NO data leakage - only past data used for indicators!


## 6. 🔐 STEP 3: Scale Data (FIT ONLY ON TRAINING!)

In [6]:
# Select features
feature_cols = [col for col in df_train.columns if col not in [config.data.date_column]]
target_col = config.data.target_column

print(f"Features: {len(feature_cols)}")
print(f"Target: {target_col}")

# Extract arrays
train_features = df_train[feature_cols].values
train_target = df_train[target_col].values.reshape(-1, 1)

val_features = df_val[feature_cols].values
val_target = df_val[target_col].values.reshape(-1, 1)

test_features = df_test[feature_cols].values
test_target = df_test[target_col].values.reshape(-1, 1)

# Scale features - FIT ONLY ON TRAINING DATA!
print("\n🔒 Fitting scalers ONLY on training data...")
feature_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaler = MinMaxScaler(feature_range=(0, 1))

scaled_train_features = feature_scaler.fit_transform(train_features)  # FIT
scaled_train_target = target_scaler.fit_transform(train_target)  # FIT

scaled_val_features = feature_scaler.transform(val_features)  # TRANSFORM ONLY
scaled_val_target = target_scaler.transform(val_target)  # TRANSFORM ONLY

scaled_test_features = feature_scaler.transform(test_features)  # TRANSFORM ONLY
scaled_test_target = target_scaler.transform(test_target)  # TRANSFORM ONLY

print("\n✅ Scalers fit ONLY on training data!")
print("✅ Validation and test data transformed using training scalers")
print("✅ NO DATA LEAKAGE!")

# Save scalers
import joblib
scaler_dir = project_root / 'models' / 'scalers_no_leakage'
scaler_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(feature_scaler, scaler_dir / 'feature_scaler.pkl')
joblib.dump(target_scaler, scaler_dir / 'target_scaler.pkl')
print(f"\n💾 Scalers saved to: {scaler_dir}")

Features: 40
Target: Close

🔒 Fitting scalers ONLY on training data...

✅ Scalers fit ONLY on training data!
✅ Validation and test data transformed using training scalers
✅ NO DATA LEAKAGE!

💾 Scalers saved to: c:\Users\USER\OneDrive\Documents\GitHub\Research-Project\backend\lstm_stock_prediction\models\scalers_no_leakage


## 7. Create Sequences

In [7]:
def create_sequences(features, target, sequence_length):
    X, y = [], []
    for i in range(sequence_length, len(features)):
        X.append(features[i-sequence_length:i])
        y.append(target[i])
    return np.array(X), np.array(y)

sequence_length = config.preprocessing.sequence_length

X_train, y_train = create_sequences(scaled_train_features, scaled_train_target, sequence_length)
X_val, y_val = create_sequences(scaled_val_features, scaled_val_target, sequence_length)
X_test, y_test = create_sequences(scaled_test_features, scaled_test_target, sequence_length)

print("=" * 70)
print("SEQUENCE SHAPES")
print("=" * 70)
print(f"Training sequences:   X={X_train.shape}, y={y_train.shape}")
print(f"Validation sequences: X={X_val.shape}, y={y_val.shape}")
print(f"Test sequences:       X={X_test.shape}, y={y_test.shape}")

# Verify shapes are correct
expected_X_shape = f"(samples, {sequence_length}, {X_train.shape[2]})"
expected_y_shape = "(samples, 1)"
print(f"\n✅ Expected X shape: {expected_X_shape}")
print(f"✅ Expected y shape: {expected_y_shape}")

if X_train.ndim == 3 and y_train.ndim == 2:
    print("\n✅ All shapes are correct!")
else:
    print(f"\n❌ ERROR: Incorrect dimensions!")
    print(f"   X_train.ndim = {X_train.ndim} (should be 3)")
    print(f"   y_train.ndim = {y_train.ndim} (should be 2)")

SEQUENCE SHAPES
Training sequences:   X=(137, 30, 40), y=(137, 1)
Validation sequences: X=(0,), y=(0,)
Test sequences:       X=(0,), y=(0,)

✅ Expected X shape: (samples, 30, 40)
✅ Expected y shape: (samples, 1)

✅ All shapes are correct!


## 8. Build Model

In [8]:
input_shape = (X_train.shape[1], X_train.shape[2])
print(f"Input shape: {input_shape}")

lstm_model_builder = LSTMStockModel(input_shape=input_shape)
model = lstm_model_builder.build_deep_lstm(
    units=config.model.lstm_units,
    dropout=config.model.dropout_rate
)

optimizer = keras.optimizers.Adam(learning_rate=config.model.learning_rate)
model.compile(
    optimizer=optimizer,
    loss='huber',
    metrics=['mae', 'mape', keras.metrics.RootMeanSquaredError(name='rmse')]
)

model.summary()

Input shape: (30, 40)


INFO:src.model:Built deep LSTM model with layers: [128, 64, 32]


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 128)        │        86,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 30, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 30, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 30, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 150,849 (589.25 KB)

 Trainable params: 150,401 (587.50 KB)

 Non-trainable params: 448 (1.75 KB)

## 9. Setup Callbacks

In [9]:
model_dir = project_root / 'models' / 'saved_models_no_leakage'
checkpoint_dir = project_root / 'models' / 'checkpoints_no_leakage'
log_dir = project_root / 'logs' / f'no_leakage_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

model_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

callback_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(filepath=str(checkpoint_dir / 'best_model.keras'), monitor='val_loss', 
                            save_best_only=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1),
    callbacks.CSVLogger(filename=str(log_dir / 'training_log.csv'))
]

print("Callbacks configured")

Callbacks configured


## 10. Train Model

In [10]:
print("\n" + "=" * 60)
print("TRAINING WITH NO DATA LEAKAGE")
print("=" * 60)
print(f"Epochs: {config.training.epochs}")
print(f"Batch size: {config.training.batch_size}")
print(f"\n Validation data has NEVER been seen by scalers or feature engineerin\n")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=config.training.epochs,
    batch_size=config.training.batch_size,
    callbacks=callback_list,
    verbose=1,
    shuffle=False  # Don't shuffle time series data
)

print("\n✓ Training completed")


TRAINING WITH NO DATA LEAKAGE
Epochs: 100
Batch size: 32

 Validation data has NEVER been seen by scalers or feature engineerin

Epoch 1/100
4/5 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.9861 - mae: 1.0675 - mape: 900.0427 - rmse: 1.2814

ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("data:0", shape=(32,), dtype=float32). Expected shape (None, 30, 40), but input has incompatible shape (32,)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(32,), dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>

## 11. Evaluate Performance

In [ ]:
# Make predictions
y_pred_train = model.predict(X_train, verbose=0)
y_pred_val = model.predict(X_val, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

# Inverse transform
y_train_actual = target_scaler.inverse_transform(y_train)
y_pred_train_actual = target_scaler.inverse_transform(y_pred_train)

y_val_actual = target_scaler.inverse_transform(y_val)
y_pred_val_actual = target_scaler.inverse_transform(y_pred_val)

y_test_actual = target_scaler.inverse_transform(y_test)
y_pred_test_actual = target_scaler.inverse_transform(y_pred_test)

# Calculate metrics
def calc_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, mape, r2

train_rmse, train_mae, train_mape, train_r2 = calc_metrics(y_train_actual, y_pred_train_actual)
val_rmse, val_mae, val_mape, val_r2 = calc_metrics(y_val_actual, y_pred_val_actual)
test_rmse, test_mae, test_mape, test_r2 = calc_metrics(y_test_actual, y_pred_test_actual)

# Display results
print("\n" + "=" * 80)
print("PERFORMANCE EVALUATION (NO DATA LEAKAGE)")
print("=" * 80)

print(f"\n{'Metric':<15} {'Training':<15} {'Validation':<15} {'Test':<15}")
print("-" * 80)
print(f"{'RMSE':<15} {train_rmse:<15.4f} {val_rmse:<15.4f} {test_rmse:<15.4f}")
print(f"{'MAE':<15} {train_mae:<15.4f} {val_mae:<15.4f} {test_mae:<15.4f}")
print(f"{'MAPE (%)':<15} {train_mape:<15.2f} {val_mape:<15.2f} {test_mape:<15.2f}")
print(f"{'R² Score':<15} {train_r2:<15.4f} {val_r2:<15.4f} {test_r2:<15.4f}")

print("\n" + "=" * 80)

# Interpret results
r2_diff = abs(train_r2 - val_r2)

if val_r2 >= 0.7 and test_r2 >= 0.7:
    print("🎉 EXCELLENT! Model generalizes well.")
elif val_r2 >= 0.5:
    print("✅ GOOD performance on validation set.")
elif val_r2 >= 0:    print("⚡ MODERATE performance. Consider improvements.")
else:
    print("❌ WARNING: Negative R² on validation.")
    print("   Model performs worse than predicting the mean.")

if r2_diff > 0.3:
    print(f"\n⚠️ Large gap between train ({train_r2:.3f}) and val ({val_r2:.3f}) R²")
    print("   Possible overfitting - increase regularization")
else:
    print(f"\n✅ Good generalization (R² gap: {r2_diff:.3f})")

# Save metrics
metrics_df = pd.DataFrame({
    'Train': [train_rmse, train_mae, train_mape, train_r2],
    'Validation': [val_rmse, val_mae, val_mape, val_r2],
    'Test': [test_rmse, test_mae, test_mape, test_r2]
}, index=['RMSE', 'MAE', 'MAPE', 'R2'])

metrics_df.to_csv(log_dir / 'metrics.csv')
print(f"\n💾 Metrics saved to: {log_dir}")

## 12. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Training history
axes[0, 0].plot(history.history['loss'], label='Train Loss')
axes[0, 0].plot(history.history['val_loss'], label='Val Loss')
axes[0, 0].set_title('Training History')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Predictions vs Actual
axes[0, 1].plot(y_test_actual[:100], label='Actual', marker='o')
axes[0, 1].plot(y_pred_test_actual[:100], label='Predicted', marker='s')
axes[0, 1].set_title(f'Test Predictions (R²={test_r2:.3f})')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Scatter plot
axes[1, 0].scatter(y_test_actual, y_pred_test_actual, alpha=0.5)
min_val = min(y_test_actual.min(), y_pred_test_actual.min())
max_val = max(y_test_actual.max(), y_pred_test_actual.max())
axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[1, 0].set_title('Predicted vs Actual')
axes[1, 0].set_xlabel('Actual')
axes[1, 0].set_ylabel('Predicted')
axes[1, 0].grid(True)

# R² comparison
r2_scores = [train_r2, val_r2, test_r2]
axes[1, 1].bar(['Train', 'Val', 'Test'], r2_scores, color=['blue', 'orange', 'green'])
axes[1, 1].axhline(y=0.7, color='r', linestyle='--', label='Good (0.7)')
axes[1, 1].axhline(y=0.8, color='g', linestyle='--', label='Excellent (0.8)')
axes[1, 1].set_title('R² Score Comparison')
axes[1, 1].set_ylabel('R²')
axes[1, 1].legend()
axes[1, 1].grid(True, axis='y')

plt.tight_layout()
plt.savefig(log_dir / 'evaluation.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Plots saved to: {log_dir}")

## 13. Summary

In [ ]:
print("\n" + "=" * 70)
print("FINAL SUMMARY (NO DATA LEAKAGE)")
print("=" * 70)

print(f"\n📊 Model: Deep LSTM {config.model.lstm_units}")
print(f"📊 Features: {X_train.shape[2]} indicators")
print(f"📊 Sequence: {config.preprocessing.sequence_length} timesteps")

print(f"\n📈 Performance:")
print(f"   Train R²: {train_r2:.4f}")
print(f"   Val R²:   {val_r2:.4f} {'✅' if val_r2 >= 0.7 else '⚠️'}")
print(f"   Test R²:  {test_r2:.4f} {'✅' if test_r2 >= 0.7 else '⚠️'}")

print(f"\n🔒 Data Integrity:")
print(f"   ✅ Split data BEFORE feature engineering")
print(f"   ✅ Scalers fit ONLY on training data")
print(f"   ✅ NO data leakage between sets")
print(f"   ✅ Proper time-series validation")

if val_r2 < 0.7:
    print(f"\n💡 To improve validation R²:")
    print(f"   1. Train longer (more epochs)")
    print(f"   2. Reduce dropout if underfitting")
    print(f"   3. Increase dropout if overfitting")
    print(f"   4. Try different sequence lengths")
    print(f"   5. Add more training data")
    print(f"   6. Simplify model if overfitting")

print("\n✓ Training completed successfully!")
print("=" * 70)